In [1]:
from pathlib import Path
import pickle

import pandas as pd
from IPython.display import display


# Buscar el archivo tanto si el notebook se ejecuta desde la raiz como desde su carpeta.
possible_paths = [
    Path("Resultados/pipeline_imputation_NPK/historico_resultados.pkl"),
    Path("../../Resultados/pipeline_imputation_NPK/historico_resultados.pkl"),
]

pickle_path = next((path for path in possible_paths if path.exists()), None)
if pickle_path is None:
    raise FileNotFoundError(
        "No se encontro historico_resultados.pkl en las rutas esperadas: "
        + ", ".join(str(path) for path in possible_paths)
    )

with pickle_path.open("rb") as file:
    historico_resultados = pickle.load(file)

print(f"Archivo cargado: {pickle_path}")
print(f"Elementos disponibles: {list(historico_resultados)}")

Archivo cargado: ../../Resultados/pipeline_imputation_NPK/historico_resultados.pkl
Elementos disponibles: ['Nitrogen', 'Phosphorus', 'Potassium']


In [2]:
metric_name = "acc_test"
records = []

for element, iterations in historico_resultados.items():
    for iteration, models in iterations.items():
        for model_name, result in models.items():
            if metric_name not in result:
                continue

            records.append(
                {
                    "Elemento": element,
                    "Iteracion": iteration,
                    "Modelo": model_name,
                    "Accuracy": float(result["acc_test"]),
                    "F1": float(result["f1_test_macro"]),
                }
            )
metrics_df = pd.DataFrame(records)
if metrics_df.empty:
    raise ValueError("No se encontraron resultados con la metrica 'acc_test'.")

summary_df = (
    metrics_df.groupby(["Modelo", "Elemento"])["Accuracy"]
    .agg(media="mean", desviacion="std", n="count")
    .reset_index()
)

# La desviacion estandar de una sola observacion se reporta como cero.
summary_df["desviacion"] = summary_df["desviacion"].fillna(0)
summary_df["Accuracy (%)"] = summary_df.apply(
    lambda row: f"{row['media'] * 100:.2f} ± {row['desviacion'] * 100:.2f}",
    axis=1,
)

report_df = (
    summary_df.pivot(index="Modelo", columns="Elemento", values="Accuracy (%)")
    .reindex(columns=["Nitrogen", "Phosphorus", "Potassium"])
    .rename_axis(None, axis=1)
    .reset_index()
)

print(f"Iteraciones procesadas: {metrics_df['Iteracion'].nunique()}")
print(f"Registros procesados: {len(metrics_df)}")
display(report_df)

Iteraciones procesadas: 20
Registros procesados: 300


,Modelo,Nitrogen,Phosphorus,Potassium
0,KNN,71.84 ± 1.87,73.16 ± 1.69,74.29 ± 1.94
1,MLP,71.44 ± 3.15,72.03 ± 2.87,73.62 ± 2.67
2,RF,79.26 ± 1.99,79.78 ± 1.73,81.41 ± 1.89
3,SVM,75.27 ± 1.64,75.97 ± 1.78,77.12 ± 1.83
4,XGB,80.54 ± 1.24,80.77 ± 1.53,82.42 ± 1.50


In [3]:
# Tabla numerica para analisis posterior.
report_numeric_df = summary_df.pivot(
    index="Modelo",
    columns="Elemento",
    values=["media", "desviacion", "n"],
)

report_csv_path = pickle_path.with_name("accuracy_report.csv")
report_df.to_csv(report_csv_path, index=False)

print(f"Reporte guardado en: {report_csv_path}")
display(report_numeric_df)

Reporte guardado en: ../../Resultados/pipeline_imputation_NPK/accuracy_report.csv


media                      desviacion                       \
Elemento  Nitrogen Phosphorus Potassium   Nitrogen Phosphorus Potassium   
Modelo                                                                    
KNN       0.718409   0.731591  0.742857   0.018749   0.016947  0.019400   
MLP       0.714433   0.720250  0.736156   0.031467   0.028679  0.026652   
RF        0.792563   0.797791  0.814065   0.019870   0.017302  0.018890   
SVM       0.752725   0.759720  0.771208   0.016378   0.017771  0.018266   
XGB       0.805376   0.807658  0.824153   0.012361   0.015346  0.015022   

                n                       
Elemento Nitrogen Phosphorus Potassium  
Modelo                                  
KNN          20.0       20.0      20.0  
MLP          20.0       20.0      20.0  
RF           20.0       20.0      20.0  
SVM          20.0       20.0      20.0  
XGB          20.0       20.0      20.0

In [4]:
f1_summary_df = (
    metrics_df.groupby(["Modelo", "Elemento"])["F1"]
    .agg(media="mean", desviacion="std", n="count")
    .reset_index()
)

f1_summary_df["desviacion"] = f1_summary_df["desviacion"].fillna(0)

f1_summary_df["F1 Macro (%)"] = f1_summary_df.apply(
    lambda row: (
        f"{row['media'] * 100:.2f} ± "
        f"{row['desviacion'] * 100:.2f}"
    ),
    axis=1,
)

f1_report_df = (
    f1_summary_df.pivot(
        index="Modelo",
        columns="Elemento",
        values="F1 Macro (%)",
    )
    .reindex(columns=["Nitrogen", "Phosphorus", "Potassium"])
    .rename_axis(None, axis=1)
    .reset_index()
)

display(f1_report_df)

f1_report_df.to_csv(
    pickle_path.with_name("f1_macro_report.csv"),
    index=False,
)

,Modelo,Nitrogen,Phosphorus,Potassium
0,KNN,71.84 ± 1.89,73.13 ± 1.74,74.23 ± 1.96
1,MLP,71.41 ± 3.16,72.01 ± 2.87,73.58 ± 2.67
2,RF,79.25 ± 1.98,79.77 ± 1.74,81.39 ± 1.90
3,SVM,75.26 ± 1.63,75.95 ± 1.78,77.10 ± 1.82
4,XGB,80.54 ± 1.23,80.76 ± 1.54,82.41 ± 1.51


In [7]:
cuartiles_paths = [
    Path("Resultados/pipeline_cuartiles/historico_progress.pkl"),
    Path("../Resultados/pipeline_cuartiles/historico_progress.pkl"),
    Path("../../Resultados/pipeline_cuartiles/historico_progress.pkl"),
]

cuartiles_pickle_path = next(
    (path for path in cuartiles_paths if path.is_file()),
    None,
)

if cuartiles_pickle_path is None:
    raise FileNotFoundError(
        "No se encontro historico_progress.pkl. Rutas revisadas: "
        + ", ".join(str(path) for path in cuartiles_paths)
    )

with cuartiles_pickle_path.open("rb") as file:
    historico_cuartiles = pickle.load(file)

cuartiles_records = []

for iteration, models in historico_cuartiles.items():
    for model_name, result in models.items():
        required_metrics = {"accuracy_test", "f1_test_macro"}
        missing_metrics = required_metrics.difference(result)
        if missing_metrics:
            raise KeyError(
                f"Faltan metricas {sorted(missing_metrics)} en "
                f"la iteracion {iteration}, modelo {model_name}."
            )

        cuartiles_records.append(
            {
                "Iteracion": iteration,
                "Modelo": model_name,
                "Accuracy": float(result["accuracy_test"]),
                "F1 Macro": float(result["f1_test_macro"]),
            }
        )

cuartiles_metrics_df = pd.DataFrame(cuartiles_records)
if cuartiles_metrics_df.empty:
    raise ValueError("historico_progress.pkl no contiene resultados de cuartiles.")

cuartiles_summary_df = (
    cuartiles_metrics_df.groupby("Modelo")
    .agg(
        Accuracy_media=("Accuracy", "mean"),
        Accuracy_desviacion=("Accuracy", "std"),
        F1_media=("F1 Macro", "mean"),
        F1_desviacion=("F1 Macro", "std"),
    )
    .fillna(0)
)

cuartiles_report_df = pd.DataFrame({
    "Modelo": cuartiles_summary_df.index,
    "Accuracy (%)": [
        f"{mean * 100:.2f} ± {std * 100:.2f}"
        for mean, std in zip(
            cuartiles_summary_df["Accuracy_media"],
            cuartiles_summary_df["Accuracy_desviacion"],
        )
    ],
    "F1 Macro (%)": [
        f"{mean * 100:.2f} ± {std * 100:.2f}"
        for mean, std in zip(
            cuartiles_summary_df["F1_media"],
            cuartiles_summary_df["F1_desviacion"],
        )
    ],
})

display(cuartiles_report_df)

cuartiles_report_path = cuartiles_pickle_path.with_name("cuartiles_report.csv")
cuartiles_report_df.to_csv(cuartiles_report_path, index=False)
print(f"Reporte guardado en: {cuartiles_report_path}")

,Modelo,Accuracy (%),F1 Macro (%)
0,KNN,81.45 ± 2.42,81.44 ± 2.43
1,MLP,79.65 ± 4.09,79.62 ± 4.14
2,RF,84.87 ± 1.87,84.85 ± 1.88
3,SVM,83.56 ± 1.82,83.55 ± 1.82
4,XGB,85.99 ± 1.42,85.98 ± 1.43


Reporte guardado en: ../../Resultados/pipeline_cuartiles/cuartiles_report.csv
